[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch3/lesson3.ipynb)

# Índices de vegetación y agua

¿Está verde? ¿Está húmedo? En esta lección calcularemos y visualizaremos el Índice de Vegetación de Diferencia Normalizada (NDVI) y el Índice de Agua de Diferencia Normalizada (NDWI) a partir de imágenes satelitales.

In [ ]:
import folium
import ee
import geemap
from datetime import datetime, timedelta

# Autenticar tu cuenta de Google con Earth Engine
ee.Authenticate()

# Escribe aquí el ID de tu proyecto, entre comillas
ee.Initialize(project="emerge-lessons")

In [ ]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """
    Agrega a un mapa de Folium una capa de teselas
    procedente de una imagen de Earth Engine.
    """
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)

    folium.raster_layers.TileLayer(
        tiles=map_id_dict["tile_fetcher"].url_format,
        attr=(
            'Map Data &copy; '
            '<a href="https://earthengine.google.com/">'
            "Google Earth Engine</a>"
        ),
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

Dibuja un círculo alrededor de un punto de entrada. Esta región se utilizará para recortar la imagen satelital.

In [ ]:
# Región de interés
point = ee.Geometry.Point(-81.660044, 28.473813)
region = point.buffer(distance=100000)

Obtén imágenes del satélite Sentinel-2 que coincidan con nuestra ubicación y el intervalo de fechas, con menos de un 10 % de cobertura de nubes.

In [ ]:
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate("2024-04-01", "2024-06-01")
    .filter(
        ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 10)
    )
)

Calcula el valor mediano de las imágenes de la colección. Esto ayuda a eliminar datos no deseados, como las nubes.

In [ ]:
image = collection.median().clip(region)

Calcula el NDVI, o Índice de Vegetación de Diferencia Normalizada, a partir de la imagen. Este índice ayuda a mostrar el estado y la densidad de la vegetación.

- Un NDVI alto indica una mayor presencia de vegetación verde.
- Fórmula: `(NIR - RED) / (NIR + RED)`

In [ ]:
ndvi = image.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

Calcula el NDWI, o Índice de Agua de Diferencia Normalizada, a partir de la imagen. Este índice ayuda a identificar la presencia de agua.

- Un NDWI alto indica una mayor presencia de agua.
- Fórmula: `(GREEN - NIR) / (GREEN + NIR)`

In [ ]:
ndwi = image.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

Configura un mapa interactivo y céntralo sobre Florida.

In [ ]:
map = folium.Map(
    location=[28.473813, -81.660044],
    tiles="Cartodb dark_matter",
    zoom_start=8
)

Agrega al mapa la capa de NDVI, que representa la vegetación. Los valores altos se mostrarán en verde.

In [ ]:
map.add_ee_layer(
    ndvi,
    {
        "min": 0,
        "max": 1,
        "palette": ["white", "green"]
    },
    "NDVI"
)

Agrega al mapa la capa de NDWI, que representa el agua. Los valores altos se mostrarán en azul.

In [ ]:
map.add_ee_layer(
    ndwi,
    {
        "min": -1,
        "max": 1,
        "palette": ["white", "blue"]
    },
    "NDWI"
)

Muestra el mapa con un control de capas en la esquina superior derecha. Este control permite mostrar u ocultar cada capa.

In [ ]:
folium.LayerControl(
    collapsed=False
).add_to(map)

display(map)

En el mapa interactivo, haz clic en la casilla junto a `NDWI` para ocultar esa capa y ver el `NDVI` que se encuentra debajo. Vuelve a hacer clic en la casilla para mostrar nuevamente el `NDWI`.